# Goodgorithm — Political-content nearest-centroid signal

Computes two mean word-vectors (political / non-political) from the hand-labelled political/civic-tone evaluation set (`training/political_tone_eval.jsonl`), for a nearest-centroid political-content signal that runs alongside the trained classifier (`political_classifier.ipynb`). A post is scored by comparing its own mean word-vector's cosine similarity to each centroid.

**Not a model in the trained-classifier sense** — no fitting beyond averaging. The live signal in `processing/` needs the same embeddings loaded at inference time (to embed each *incoming* post the same way), so the artifact this notebook publishes is small (two 100-dimensional vectors), but the runtime dependency it implies is not.

**No GPU needed.** Uses the same 60% training split (fixed seed) as `political_classifier.ipynb`, so the two signals are comparable and combinable at the same held-out check posts.

In [ ]:
!pip install -q gensim boto3

## Load the political/civic-tone evaluation set

In [ ]:
import json
import random
from collections import Counter, defaultdict

random.seed(42)

DATA_PATH = "political_tone_eval.jsonl"  # copy training/political_tone_eval.jsonl next to this notebook before running

rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8")]
for r in rows:
    r["is_political"] = r["label"] != "non-political"

by_label = defaultdict(list)
for r in rows:
    by_label[r["label"]].append(r)

train, test = [], []
for label, items in by_label.items():
    items = items[:]
    random.shuffle(items)
    cut = int(len(items) * 0.6)
    train += items[:cut]
    test += items[cut:]
random.shuffle(train)
random.shuffle(test)
print(f"train={len(train)} test(check)={len(test)}")

## Load embeddings and compute the two centroids

Plain lowercase-regex tokenization, not `text_normalize.normalize_text()` — the classifier's TF-IDF pipeline needs normalization for train/inference parity through its ONNX export, but this signal has no such export step, and the live scoring code in `processing/` must tokenize identically to however the centroids were computed here.

In [ ]:
import re

import numpy as np
import gensim.downloader as api

EMBEDDING = "glove-twitter-100"
kv = api.load(EMBEDDING)

TOKEN_RE = re.compile(r"[a-zA-Z']+")


def tokenize(text):
    return [t.lower() for t in TOKEN_RE.findall(text or "")]


def mean_vector(tokens):
    vecs = [kv[t] for t in tokens if t in kv]
    if not vecs:
        return None
    return np.mean(vecs, axis=0)


train_pol_vecs = [mean_vector(tokenize(r["text"])) for r in train if r["is_political"]]
train_nonpol_vecs = [mean_vector(tokenize(r["text"])) for r in train if not r["is_political"]]
train_pol_vecs = [v for v in train_pol_vecs if v is not None]
train_nonpol_vecs = [v for v in train_nonpol_vecs if v is not None]

political_centroid = np.mean(train_pol_vecs, axis=0)
non_political_centroid = np.mean(train_nonpol_vecs, axis=0)
print(f"centroids fit on {len(train_pol_vecs)} political / {len(train_nonpol_vecs)} non-political train posts")

## Spot-check

In [ ]:
def cosine(a, b):
    if a is None or b is None:
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def score(text):
    v = mean_vector(tokenize(text))
    return cosine(v, political_centroid) - cosine(v, non_political_centroid)


spot_checks = [
    "The senate voted 51-49 to pass the appropriations bill today",
    "Just adopted the sweetest rescue puppy, she's already best friends with the cat!",
    "Thanks Donny 👑✝️🤡 #maga #grift",
    "Tried a new ramen spot downtown and it was incredible",
    "My daughter registered to vote for the first time today, so proud",
    "New indie album dropped and it's already on repeat",
    "The White House loves 60 Minutes, per the latest reporting",
    "Woke up early, went for a long walk, feeling great today",
]
for text in spot_checks:
    print(f"  {score(text):+.4f}  {text}")

## Package `centroids.json`

In [ ]:
import json
from datetime import datetime, timezone

VERSION = "v1"

config = {
    "version": VERSION,
    "embedding": EMBEDDING,
    "embedding_dim": int(political_centroid.shape[0]),
    "political_centroid": political_centroid.tolist(),
    "non_political_centroid": non_political_centroid.tolist(),
    "dataset_composition": {
        "source": "training/political_tone_eval.jsonl (60% train split)",
        "train_political_examples": len(train_pol_vecs),
        "train_nonpolitical_examples": len(train_nonpol_vecs),
    },
    "trained_at": datetime.now(timezone.utc).isoformat(),
}

with open("centroids.json", "w") as f:
    json.dump(config, f)

print(json.dumps({k: v for k, v in config.items() if not k.endswith("centroid")}, indent=2))

## Upload to R2

Uploads always happen unconditionally — versioned artifacts are cheap and safe to publish. Promoting a version to production (flipping `latest.json`) is a separate, deliberate step, not done here.

In [ ]:
R2_MODELS_ACCOUNT_ID = R2_MODELS_ACCESS_KEY_ID = R2_MODELS_SECRET_ACCESS_KEY = R2_MODELS_BUCKET_NAME = None

try:
    from google.colab import userdata

    R2_MODELS_ACCOUNT_ID = userdata.get("R2_MODELS_ACCOUNT_ID")
    R2_MODELS_ACCESS_KEY_ID = userdata.get("R2_MODELS_ACCESS_KEY_ID")
    R2_MODELS_SECRET_ACCESS_KEY = userdata.get("R2_MODELS_SECRET_ACCESS_KEY")
    R2_MODELS_BUCKET_NAME = userdata.get("R2_MODELS_BUCKET_NAME")
except Exception as e:
    print(f"Colab secrets unavailable ({type(e).__name__}: {e}), trying Kaggle secrets...")

if not R2_MODELS_ACCOUNT_ID:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        R2_MODELS_ACCOUNT_ID = secrets.get_secret("R2_MODELS_ACCOUNT_ID")
        R2_MODELS_ACCESS_KEY_ID = secrets.get_secret("R2_MODELS_ACCESS_KEY_ID")
        R2_MODELS_SECRET_ACCESS_KEY = secrets.get_secret("R2_MODELS_SECRET_ACCESS_KEY")
        R2_MODELS_BUCKET_NAME = secrets.get_secret("R2_MODELS_BUCKET_NAME")
    except Exception as e:
        print(f"Kaggle secrets unavailable ({type(e).__name__}: {e}), falling back to manual values...")

if not R2_MODELS_ACCOUNT_ID:
    # Manual fallback -- fill these in locally, never commit real values.
    R2_MODELS_ACCOUNT_ID = ""
    R2_MODELS_ACCESS_KEY_ID = ""
    R2_MODELS_SECRET_ACCESS_KEY = ""
    R2_MODELS_BUCKET_NAME = ""

assert (
    R2_MODELS_ACCOUNT_ID
    and R2_MODELS_ACCESS_KEY_ID
    and R2_MODELS_SECRET_ACCESS_KEY
    and R2_MODELS_BUCKET_NAME
), "R2 credentials not set -- see the markdown cell above"

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_MODELS_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_MODELS_ACCESS_KEY_ID,
    aws_secret_access_key=R2_MODELS_SECRET_ACCESS_KEY,
    region_name="auto",
)

PREFIX = f"political-centroid/{VERSION}"
s3.upload_file("centroids.json", R2_MODELS_BUCKET_NAME, f"{PREFIX}/centroids.json")
print(f"uploaded to s3://{R2_MODELS_BUCKET_NAME}/{PREFIX}/")

## Promote to production

Not promoted here. Promotion is a deliberate, separate step, run once something actually loads this model:
```
cd training && uv run python r2_release.py --model political-centroid publish v1
```